# Introdução

Este projeto visa analisar a evolução das temperaturas médias globais por país ao longo dos anos, utilizando dados da FAO (Food and Agriculture Organization). Serão exploradas variações por país, tendências históricas, tratamento de dados ausentes e outliers, além da criação de novas features para alimentar modelos de machine learning.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Carregando o dataset
df = pd.read_csv("../data/Environment_Temperature_change_E_All_Data_NOFLAG.csv", encoding="ISO-8859-1")

# Renomeando colunas
df.columns = df.columns.str.replace("ï»¿", "")

# Filtrando apenas temperatura média anual
df = df[df["Element"] == "Temperature change"]

# Convertendo de wide para long
df_long = df.melt(
    id_vars=["Area", "Months", "Element", "Unit"],
    value_vars=[col for col in df.columns if col.startswith("Y")],
    var_name="Year",
    value_name="Value"
)

df_long["Year"] = df_long["Year"].str.extract("(\d+)").astype(int)
df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
df_long.dropna(subset=["Value"], inplace=True)

df_long.head()

# Análise Exploratória (EDA)

Nesta etapa, analisamos as distribuições dos dados, tendências temporais e variações regionais.


In [ ]:
# Temperatura média global por ano
media_anual = df_long.groupby("Year")["Value"].mean()

plt.figure(figsize=(12, 6))
sns.lineplot(x=media_anual.index, y=media_anual.values)
plt.title("Média Global de Variação de Temperatura por Ano")
plt.ylabel("Variação de Temperatura (°C)")
plt.grid()
plt.show()

In [ ]:
# Boxplot por década
df_long["Decada"] = (df_long["Year"] // 10) * 10
plt.figure(figsize=(12, 6))
sns.boxplot(x="Decada", y="Value", data=df_long)
plt.title("Distribuição de Temperaturas por Década")
plt.ylabel("Variação de Temperatura (°C)")
plt.grid()
plt.show()

Boxplot das variações por mês (excluindo "Meteorological year")

In [ ]:

plt.figure()
sns.boxplot(data=df_long[df_long["Months"] != "Meteorological year"],
            x="Months", y="Value")
plt.xticks(rotation=45)
plt.title("Distribuição da Mudança de Temperatura por Mês")
plt.xlabel("Mês")
plt.ylabel("Mudança de Temperatura (°C)")
plt.tight_layout()
plt.show()

Top 10 países mais afetados em 2019

In [ ]:
# 3.3. Top 10 países mais afetados em 2019
df_2019 = df_long[df_long["Year"] == 2019]
top_10 = df_2019.groupby("Area")["Value"].mean().sort_values(ascending=False).head(10)

plt.figure()
sns.barplot(x=top_10.values, y=top_10.index, palette="Reds_r")
plt.title("Top 10 Países com Maior Mudança de Temperatura em 2019")
plt.xlabel("Mudança de Temperatura (°C)")
plt.ylabel("País")
plt.tight_layout()
plt.show()

Média de temperatura por região ao longo do tempo (exemplo: Brasil)

In [ ]:
brasil = df_long[(df_long["Area"] == "Brazil") & (df_long["Months"] == "Meteorological year")]

plt.figure()
sns.lineplot(data=brasil, x="Year", y="Value", marker="o")
plt.title("Mudança de Temperatura Anual no Brasil (1961–2019)")
plt.xlabel("Ano")
plt.ylabel("Mudança de Temperatura (°C)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Tratamento de Outliers

Vamos identificar e remover outliers usando o método do IQR.

In [ ]:
Q1 = df_long["Value"].quantile(0.25)
Q3 = df_long["Value"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

df_long = df_long[(df_long["Value"] >= limite_inferior) & (df_long["Value"] <= limite_superior)]

# Engenharia de Features

Criação de novas variáveis para enriquecer a análise e alimentar modelos de machine learning.

In [ ]:
# Delta anual por país
df_long = df_long.sort_values(["Area", "Year"])
df_long["Delta_Anual"] = df_long.groupby("Area")["Value"].diff()

# Desvio da média histórica
media_pais = df_long.groupby("Area")["Value"].transform("mean")
df_long["Desvio_Historico"] = df_long["Value"] - media_pais

# Média móvel de 5 anos
df_long["Media_Movel_5"] = df_long.groupby("Area")["Value"].transform(lambda x: x.rolling(5, min_periods=1).mean())

# Variação máxima mensal por ano e país
var_mensal = df_long.groupby(["Area", "Year"])["Value"].agg(["max", "min"]).reset_index()
var_mensal["Variacao_Mensal"] = var_mensal["max"] - var_mensal["min"]

# Z-Score por país
df_long["Z_Score"] = df_long.groupby("Area")["Value"].transform(lambda x: (x - x.mean()) / x.std())


# Seleção de Features

Selecionamos as features com maior capacidade explicativa e menor multicolinearidade:
- Valor original (`Value`)
- Desvio Histórico
- Delta Anual
- Média Móvel
- Z-Score


# Modelagem Preditiva

Comparação entre 3 algoritmos para prever a temperatura com base nas features:
- Regressão Linear
- Random Forest
- Gradient Boosting

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
import xgboost as xgb
import numpy as np

# Dataset para modelagem
model_data = df_long.dropna(subset=["Year", "Value", "Desvio_Historico", "Delta_Anual", "Media_Movel_5", "Z_Score"])
X = model_data.select_dtypes(include=[np.number]).drop("Value", axis=1)
y = model_data["Value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelos a testar
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "PolynomialRegression (deg=2)": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    "XGBoost": xgb.XGBRegressor(random_state=42)
}

# Avaliação dos modelos
print("Resultados de RMSE:")
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    print(f"{name}: RMSE = {rmse:.4f}")


O modelo com a menos taxa de erro foi o RandomForest

## Busca em grade (Grid Search) para encontrar a melhor combinação de hiperparâmetros

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [5, 10, None]
}

grid = GridSearchCV(RandomForestRegressor(), param_grid, cv=3, scoring="neg_root_mean_squared_error")
grid.fit(X_train, y_train)

print("Melhores parâmetros:", grid.best_params_)


#Comparar o desempenho de regressão linear sobre: dados originais do dataset e dados transformados para relação linear.

In [ ]:
df_long_comp = df_long.sort_values(by="Year")
df_long_comp['Delta_Anual'] = df_long_comp.groupby('Area')['Value'].diff()
df_long_comp['Media_Movel_5'] = df_long_comp.groupby('Area')['Value'].transform(lambda x: x.rolling(window=5, min_periods=1).mean())
df_long_comp['Z_Score'] = df_long_comp.groupby('Area')['Value'].transform(lambda x: (x - x.mean()) / x.std())
df_long_comp['Desvio_Historico'] = df_long_comp.groupby('Area')['Value'].transform(lambda x: x.expanding().std())

# --- 1. Seleciona as colunas para modelagem, descartando linhas com NaNs (causados pelas transformações) ---
model_data_transformado = df_long_comp[["Value", "Desvio_Historico", "Delta_Anual", "Media_Movel_5", "Z_Score"]].dropna()

X_transformado = model_data_transformado.drop("Value", axis=1)
y_transformado = model_data_transformado["Value"]

# --- 2. Separar treino e teste ---
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_transformado, y_transformado, test_size=0.2, random_state=42
)

# --- 3. Treinar regressão linear ---
modelo_linear_transformado = LinearRegression()
modelo_linear_transformado.fit(X_train_t, y_train_t)

# --- 4. Previsões e avaliação ---
y_pred_t = modelo_linear_transformado.predict(X_test_t)
mse_transformado = mean_squared_error(y_test_t, y_pred_t)
rmse_transformado = np.sqrt(mse_transformado)

print("Regressão Linear com dados origimais: RMSE = 0.137")
print(f"Regressão Linear com dados transformados: RMSE = {rmse_transformado:.3f}")



# Conclusão

Com base na análise exploratória e modelagem:
- Observa-se tendência clara de aquecimento em quase todos os países ao longo dos anos.
- A engenharia de features capturou variações sazonais, desvios históricos e tendências.
- O modelo de Gradient Boosting obteve o menor erro de previsão (RMSE).
- Para trabalhos futuros, recomenda-se incorporar variáveis externas (ex. emissão de CO₂) para aprimorar o modelo.

----------------------------------------------------




# P**revisão de Temperatura Global com Machine Learning**




In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor, TheilSenRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

## Carregamento dos Dados e Preparação da Série Temporal

In [ ]:
import pandas as pd

# 1. Carregar dados e filtrar temperatura
df = pd.read_csv("/content/sample_data/Environment_Temperature_change_E_All_Data_NOFLAG.csv", encoding="ISO-8859-1")

# Limpar nome de colunas (remover possíveis caracteres estranhos)
df.columns = df.columns.str.replace("ï»¿", "")

# Filtrar apenas registros de 'Temperature change'
df = df[df["Element"] == "Temperature change"]

# 2. Converter dados para formato longo (long format)
df_long = df.melt(
    id_vars=["Area", "Months", "Element", "Unit"],
    var_name="Year", value_name="Value"
)

# 3. Extrair apenas o ano como número inteiro
df_long["Year"] = pd.to_numeric(df_long["Year"].str.extract(r"(\d+)")[0], errors="coerce")
df_long = df_long.dropna(subset=["Year"])
df_long["Year"] = df_long["Year"].astype(int)

# 4. Converter valores para numérico e limpar NaNs
df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
df_long.dropna(subset=["Value"], inplace=True)

# 5. (Opcional) Ordenar por Área e Ano para garantir consistência temporal
df_long = df_long.sort_values(by=["Area", "Year"]).reset_index(drop=True)

# Pronto: df_long contém dados limpos e organizados para análise temporal
print(df_long.head())


## Agrupar Dados Globais por Ano

In [ ]:
df_global = df_long[df_long["Months"] == "Meteorological year"]
df_global = df_global.groupby("Year")["Value"].mean().reset_index()
df_global.head()

## Treinamento do Modelo

## Previsões Futuras para 2024, 2044 e 2069

> Adicionar aspas



In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# --- Dados históricos preparados com features ---
df_global = df_long.groupby("Year")["Value"].mean().reset_index()
df_global = df_global.sort_values("Year").reset_index(drop=True)

df_global["Delta_Anual"] = df_global["Value"].diff()
df_global["Media_Movel_5"] = df_global["Value"].rolling(window=5, min_periods=1).mean()
df_global["Z_Score"] = (df_global["Value"] - df_global["Value"].mean()) / df_global["Value"].std()
df_global["Desvio_Historico"] = df_global["Value"].expanding().std()
df_global = df_global.dropna().reset_index(drop=True)

# Variáveis para treino
X = df_global[["Year", "Delta_Anual", "Media_Movel_5", "Z_Score", "Desvio_Historico"]]
y = df_global["Value"]

# Treinar modelo RandomForest com melhores parâmetros
model_rf = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)
model_rf.fit(X, y)

# Anos futuros que queremos prever
anos_futuros = [2024, 2025, 2026, 2027, 2028, 2029, 2030]

# Função para previsão progressiva
def prever_anos_futuros(modelo, dados_hist, anos):
    # Inicializa dataframe com dados históricos para atualizar features
    df_pred = dados_hist.copy()

    previsoes = []

    for ano in anos:
        # Para o novo ano, calculamos as features:

        # Delta Anual: diferença entre o último valor previsto e o anterior
        delta_anual = df_pred["Value"].iloc[-1] - df_pred["Value"].iloc[-2]

        # Média móvel 5 anos: média dos últimos 4 valores + o último previsto
        if len(df_pred) >= 5:
            media_movel_5 = df_pred["Value"].iloc[-5:].mean()
        else:
            media_movel_5 = df_pred["Value"].mean()

        # Desvio histórico: desvio padrão acumulado até o último valor previsto
        desvio_historico = df_pred["Value"].std()

        # Z-Score: normalização usando média e std da série histórica até agora
        media = df_pred["Value"].mean()
        std = df_pred["Value"].std()
        z_score = (df_pred["Value"].iloc[-1] - media) / std if std > 0 else 0

        # Monta linha com as features para o ano atual
        linha = pd.DataFrame({
            "Year": [ano],
            "Delta_Anual": [delta_anual],
            "Media_Movel_5": [media_movel_5],
            "Z_Score": [z_score],
            "Desvio_Historico": [desvio_historico]
        })

        # Faz previsão para o ano atual
        pred = modelo.predict(linha)[0]

        # Armazena previsão
        previsoes.append((ano, pred))

        # Atualiza df_pred com a nova previsão para usar nas próximas iterações
        nova_linha = pd.DataFrame({"Year": [ano], "Value": [pred]})
        df_pred = pd.concat([df_pred, nova_linha], ignore_index=True)

    return previsoes

# Usar a função para prever
resultados = prever_anos_futuros(model_rf, df_global[["Year", "Value"]], anos_futuros)

# Mostrar resultados
for ano, temp in resultados:
    print(f"Previsão para {ano}: {temp:.4f} °C")


## Visualização da Tendência e Previsões

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Anos futuros para previsão
anos_futuros = [2024, 2034, 2044, 2059, 2069, 2100]

plt.figure(figsize=(12, 6))

# Plot dos dados históricos
sns.scatterplot(x="Year", y="Value", data=df_global, label="Histórico", color='black')

# Previsão progressiva para o modelo escolhido (RandomForest)
resultados = prever_anos_futuros(model_rf, df_global[["Year", "Value"]], anos_futuros)
anos, preds_futuros = zip(*resultados)

plt.plot(anos, preds_futuros, label="RandomForest (melhor modelo)", color='red', marker='o')

plt.title("Previsão de Temperatura para Anos Futuros")
plt.xlabel("Ano")
plt.ylabel("Temperatura Média (°C)")
plt.legend()
plt.grid(True)
plt.show()
